In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import h5py
import sunpy 
import sunpy.map 
from astropy.io import fits, ascii
from astropy.wcs import WCS
from copy import deepcopy

In [2]:
import ndcube
from ndcube import NDCube
from ndcube.wcs.tools import unwrap_wcs_to_fitswcs

In [3]:
dkist_vbi_target_header = fits.getheader("../../data/pid_1_123_aux/plot_ready/dkist_target_wcs_header_before_crop.fits",
                                        ignore_missing_simple=True,)

dkist_vbi_target_data = np.zeros((4096,4096))

dkist_vbi_target_cube = NDCube(dkist_vbi_target_data,WCS(dkist_vbi_target_header, naxis=2, fix=False, relax=True, preserve_units=True))
dkist_vbi_target_cube_crop = dkist_vbi_target_cube[128:-128,128:-128]
dkist_vbi_target_cube_crop_wcs = unwrap_wcs_to_fitswcs(dkist_vbi_target_cube_crop.wcs)[0]

In [4]:
dkist_vbi_target_cube_crop_wcs

WCS Keywords

Number of WCS axes: 2
CTYPE : 'HPLN-TAN' 'HPLT-TAN'
CUNIT : 'arcsec' 'arcsec'
CRVAL : -393.24226873564993 199.8803497388929
CRPIX : 1885.3529278611363 1953.5712893851355
PC1_1 PC1_2  : 1.0375098416387665 -0.01300651478581309
PC2_1 PC2_2  : 0.0128718573348022 1.0594909081129698
CDELT : 0.01015898585319519 0.01015898585319519
NAXIS : 3840  3840

In [5]:
dkist_vbi_target_cube_crop_header = dkist_vbi_target_cube_crop_wcs.to_header()

In [6]:
dkist_vbi_target_cube_crop_header["DATE-OBS"] = (dkist_vbi_target_cube_crop_header["DATE-BEG"],
                                                 dkist_vbi_target_cube_crop_header.comments["DATE-BEG"])
dkist_vbi_target_cube_crop_header

WCSAXES =                    2 / Number of coordinate axes                      
CRPIX1  =      1885.3529278611 / Pixel coordinate of reference point            
CRPIX2  =      1953.5712893851 / Pixel coordinate of reference point            
PC1_1   =      1.0375098416388 / Coordinate transformation matrix element       
PC1_2   =   -0.013006514785813 / Coordinate transformation matrix element       
PC2_1   =    0.012871857334802 / Coordinate transformation matrix element       
PC2_2   =       1.059490908113 / Coordinate transformation matrix element       
CDELT1  =    0.010158985853195 / [arcsec] Coordinate increment at reference poin
CDELT2  =    0.010158985853195 / [arcsec] Coordinate increment at reference poin
CUNIT1  = 'arcsec'             / Units of coordinate increment and value        
CUNIT2  = 'arcsec'             / Units of coordinate increment and value        
CTYPE1  = 'HPLN-TAN'           / Coordinate type codegnomonic projection        
CTYPE2  = 'HPLT-TAN'        

In [7]:
dkist_vbi_target_cube_crop_header_with_crota2 = deepcopy(dkist_vbi_target_cube_crop_header)

dkist_vbi_target_cube_crop_header_with_crota2["CROTA2"] = (np.rad2deg(np.arctan2(-dkist_vbi_target_cube_crop_header["PC1_2"],
                                                                                dkist_vbi_target_cube_crop_header["PC1_1"]) + \
                                                                     np.arctan2(dkist_vbi_target_cube_crop_header["PC2_1"],
                                                                                dkist_vbi_target_cube_crop_header["PC2_2"]))/2,
                                                        "[deg] an approximated coordinate rotation angle as PCi_j matrix is not perfectly orthogonal")

# a bad approximation for crota2...
dkist_vbi_target_cube_crop_header_with_crota2.remove("PC1_1")
dkist_vbi_target_cube_crop_header_with_crota2.remove("PC1_2")
dkist_vbi_target_cube_crop_header_with_crota2.remove("PC2_1")
dkist_vbi_target_cube_crop_header_with_crota2.remove("PC2_2")

dkist_vbi_target_cube_crop_header_with_crota2

WCSAXES =                    2 / Number of coordinate axes                      
CRPIX1  =      1885.3529278611 / Pixel coordinate of reference point            
CRPIX2  =      1953.5712893851 / Pixel coordinate of reference point            
CDELT1  =    0.010158985853195 / [arcsec] Coordinate increment at reference poin
CDELT2  =    0.010158985853195 / [arcsec] Coordinate increment at reference poin
CUNIT1  = 'arcsec'             / Units of coordinate increment and value        
CUNIT2  = 'arcsec'             / Units of coordinate increment and value        
CTYPE1  = 'HPLN-TAN'           / Coordinate type codegnomonic projection        
CTYPE2  = 'HPLT-TAN'           / Coordinate type codegnomonic projection        
CRVAL1  =     -393.24226873565 / [arcsec] Coordinate value at reference point   
CRVAL2  =      199.88034973889 / [arcsec] Coordinate value at reference point   
LONPOLE =                180.0 / [deg] Native longitude of celestial pole       
LATPOLE =    0.0555223193719

In [8]:
data_empty = np.zeros((4096-128*2,4096-128*2), dtype=np.int8)

In [9]:
dkist_vbi_target_cube_crop_hdu = fits.PrimaryHDU(data_empty, header=dkist_vbi_target_cube_crop_header)
dkist_vbi_target_cube_crop_hdu_with_crota2 = fits.PrimaryHDU(data_empty, header=dkist_vbi_target_cube_crop_header_with_crota2)

In [10]:
dkist_vbi_target_cube_crop_hdul_with_crota2 = fits.HDUList([dkist_vbi_target_cube_crop_hdu_with_crota2])

In [11]:
dkist_vbi_target_cube_crop_hdul_with_crota2.writeto("../../data/pid_1_123_aux/dkist_20221024_sudip/dkist_target_wcs_header_after_crop_with_crota2.fits", overwrite=True)

In [12]:
test_hdr = fits.getheader("../../data/pid_1_123_aux/dkist_20221024_sudip/dkist_target_wcs_header_after_crop_with_crota2.fits", ignore_missing_simple=True,)

In [13]:
test_hdr

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 3840                                                  
NAXIS2  =                 3840                                                  
WCSAXES =                    2 / Number of coordinate axes                      
CRPIX1  =      1885.3529278611 / Pixel coordinate of reference point            
CRPIX2  =      1953.5712893851 / Pixel coordinate of reference point            
PC1_1   =      1.0375098416388 / Coordinate transformation matrix element       
PC1_2   =   -0.013006514785813 / Coordinate transformation matrix element       
PC2_1   =    0.012871857334802 / Coordinate transformation matrix element       
PC2_2   =       1.059490908113 / Coordinate transformation matrix element       
CDELT1  =    0.0101589858531

In [14]:
file_Gband_pr = h5py.File("../../data/pid_1_123_aux/plot_ready/Gband_AEZDV_pr.hdf5")
Gband_pr_set = file_Gband_pr["vbi_img"]
Gband_date_obs = ascii.read("../../data/pid_1_123_aux/plot_ready/Gband_AEZDV_date_avg.txt")


In [15]:
Gband_pr_set.dtype

dtype('<f8')

In [16]:
Gband_hdul = fits.HDUList(
    [
        dkist_vbi_target_cube_crop_hdu,
        fits.ImageHDU(Gband_pr_set.astype(np.float32), name="DATA_CUBE"),
        fits.BinTableHDU(Gband_date_obs, name="DATE_OBS"),
    ]
)

In [17]:
Gband_hdul.writeto("../../data/pid_1_123_aux/dkist_20221024_sudip/Gband_AEZDV_20221024.fits", overwrite=True)

In [18]:
test_hdr = fits.getheader("../../data/pid_1_123_aux/dkist_20221024_sudip/Gband_AEZDV_20221024.fits", ignore_missing_simple=True,ext=0)

In [19]:
test_hdr

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 3840                                                  
NAXIS2  =                 3840                                                  
EXTEND  =                    T                                                  
WCSAXES =                    2 / Number of coordinate axes                      
CRPIX1  =      1885.3529278611 / Pixel coordinate of reference point            
CRPIX2  =      1953.5712893851 / Pixel coordinate of reference point            
PC1_1   =      1.0375098416388 / Coordinate transformation matrix element       
PC1_2   =   -0.013006514785813 / Coordinate transformation matrix element       
PC2_1   =    0.012871857334802 / Coordinate transformation matrix element       
PC2_2   =       1.0594909081

In [20]:
del Gband_hdul

In [21]:
file_Hbeta_pr = h5py.File("../../data/pid_1_123_aux/plot_ready/Hbeta_BJOLO_pr.hdf5")
Hbeta_pr_set = file_Hbeta_pr["vbi_img"]
Hbeta_date_obs = ascii.read("../../data/pid_1_123_aux/plot_ready/Hbeta_BJOLO_date_avg.txt")
Hbeta_hdul = fits.HDUList(
    [
        dkist_vbi_target_cube_crop_hdu,
        fits.ImageHDU(Hbeta_pr_set.astype(np.float32), name="DATA_CUBE"),
        fits.BinTableHDU(Hbeta_date_obs, name="DATE_OBS"),
    ]
)
Hbeta_hdul.writeto("../../data/pid_1_123_aux/dkist_20221024_sudip/Hbeta_BJOLO_20221024.fits", overwrite=True)

del Hbeta_hdul

In [ ]:
file_CaIIK_pr = h5py.File("../../data/pid_1_123_aux/plot_ready/CaIIK_BZPOW_pr.hdf5")
CaIIK_pr_set = file_CaIIK_pr["vbi_img"]
CaIIK_date_obs = ascii.read("../../data/pid_1_123_aux/plot_ready/CaIIK_BZPOW_date_avg.txt")
CaIIK_hdul = fits.HDUList(
    [
        dkist_vbi_target_cube_crop_hdu,
        fits.ImageHDU(CaIIK_pr_set.astype(np.float32), name="DATA_CUBE"),
        fits.BinTableHDU(CaIIK_date_obs, name="DATE_OBS"),
    ]
)
CaIIK_hdul.writeto("../../data/pid_1_123_aux/dkist_20221024_sudip/CaIIK_BZPOW_20221024.fits", overwrite=True)

del CaIIK_hdul

In [ ]:
file_TiO_pr = h5py.File("../../data/pid_1_123_aux/plot_ready/TiO_BNRPZ_pr.hdf5")
TiO_pr_set = file_TiO_pr["vbi_img"]
TiO_date_obs = ascii.read("../../data/pid_1_123_aux/plot_ready/TiO_BNRPZ_date_avg.txt")
TiO_hdul = fits.HDUList(
    [
        dkist_vbi_target_cube_crop_hdu,
        fits.ImageHDU(TiO_pr_set.astype(np.float32), name="DATA_CUBE"),
        fits.BinTableHDU(TiO_date_obs, name="DATE_OBS"),
    ]
)
TiO_hdul.writeto("../../data/pid_1_123_aux/dkist_20221024_sudip/TiO_BNRPZ_20221024.fits", overwrite=True)

del TiO_hdul

In [ ]:
file_Halpha_pr = h5py.File("../../data/pid_1_123_aux/plot_ready/Halpha_BLZNL_pr.hdf5")
Halpha_pr_set = file_Halpha_pr["vbi_img"]
Halpha_date_obs = ascii.read("../../data/pid_1_123_aux/plot_ready/Halpha_BLZNL_date_avg.txt")
Halpha_hdul = fits.HDUList(
    [
        dkist_vbi_target_cube_crop_hdu,
        fits.ImageHDU(Halpha_pr_set.astype(np.float32), name="DATA_CUBE"),
        fits.BinTableHDU(Halpha_date_obs, name="DATE_OBS"),
    ]
)
Halpha_hdul.writeto("../../data/pid_1_123_aux/dkist_20221024_sudip/Halpha_BLZNL_20221024.fits", overwrite=True)

del Halpha_hdul